# Setup

In [ ]:
!pip install transformers

In [ ]:
!pip install torch

In [ ]:
import torch
device =torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
model_name="google/functiongemma-270m-it"

In [ ]:
tokenizer= AutoTokenizer.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/63.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

## Test tokenizer

In [ ]:
# tokenizer("Hello world")

In [ ]:
input_conversation_example=[
    {"role": "user", "content": "Classify the email as SPAM or HAM and explain why.\n\nEmail: Congratulations! You have won a ₹10,000 Amazon gift card. Click here to claim now!"},
    {"role": "assistant", "content": """{"label": "SPAM", "reason": "Unsolicited reward with urgent call to action"}"""}
    ]

In [ ]:
input_tokens= tokenizer.apply_chat_template(
    conversation=input_conversation_example,
    tokenize=True,
    ).to(device)
input_tokens

{'input_ids': [2, 105, 2364, 107, 4335, 1891, 506, 4553, 618, 11446, 1936, 653, 88007, 532, 8082, 3217, 236761, 108, 11919, 236787, 89099, 236888, 1599, 735, 2810, 496, 31386, 236770, 236771, 236764, 236771, 236771, 236771, 9593, 7963, 3733, 236761, 11738, 1590, 531, 3539, 1492, 236888, 106, 107, 105, 4368, 107, 14937, 2491, 1083, 623, 6120, 1936, 827, 623, 27388, 1083, 623, 2805, 6169, 152718, 13833, 607, 32034, 2246, 531, 2970, 25938, 106, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

# Load Model

In [ ]:
import torch
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
).to(device)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

# Test if model works? - Before Fine Tuning

In [ ]:
input="Classify the email as SPAM or HAM and explain why.\n\nEmail: Congratulations! You have won a ₹10,000 Amazon gift card. Click here to claim now!"
i_tokens=tokenizer(input,return_tensors="pt")["input_ids"].to(device)
output_tokens=model.generate(i_tokens)
tokenizer.decode(output_tokens)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['<bos>Classify the email as SPAM or HAM and explain why.\n\nEmail: Congratulations! You have won a ₹10,000 Amazon gift card. Click here to claim now!\n\n**Classification:** SPAM\n\n**Reason:**\n\n1.  **Classification:** SPAM (']

# Fine tuning starts

In [ ]:
input_conversation=[
    {"role": "user", "content": "Classify the email as SPAM or HAM and explain why.\n\nEmail: Congratulations! You have won a ₹10,000 Amazon gift card. Click here to claim now!"},
    {"role": "assistant", "content": '{ "label": "SPAM", "reason": "'}
    ]

In [ ]:
input_detokens= tokenizer.apply_chat_template(
    conversation=input_conversation,
    tokenize=False,
    continue_final_message=True,
    )
input_detokens

'<bos><start_of_turn>user\nClassify the email as SPAM or HAM and explain why.\n\nEmail: Congratulations! You have won a ₹10,000 Amazon gift card. Click here to claim now!<end_of_turn>\n<start_of_turn>model\n{ "label": "SPAM", "reason": "'

In [ ]:
output_label="Unsolicited reward with urgency." + '" }'
full_conversation=input_detokens+output_label+tokenizer.eos_token
full_conversation

'<bos><start_of_turn>user\nClassify the email as SPAM or HAM and explain why.\n\nEmail: Congratulations! You have won a ₹10,000 Amazon gift card. Click here to claim now!<end_of_turn>\n<start_of_turn>model\n{ "label": "SPAM", "reason": "Unsolicited reward with urgency." }<eos>'

In [ ]:
input_tokenized= tokenizer(full_conversation, return_tensors="pt", add_special_tokens=False).to(device)["input_ids"]
input_tokenized

tensor([[     2,    105,   2364,    107,   4335,   1891,    506,   4553,    618,
          11446,   1936,    653,  88007,    532,   8082,   3217, 236761,    108,
          11919, 236787,  89099, 236888,   1599,    735,   2810,    496,  31386,
         236770, 236771, 236764, 236771, 236771, 236771,   9593,   7963,   3733,
         236761,  11738,   1590,    531,   3539,   1492, 236888,    106,    107,
            105,   4368,    107, 236782,    623,   2491,   1083,    623,   6120,
           1936,    827,    623,  27388,   1083,    623,   2805,   6169, 152718,
          13833,    607,  60055,   1781,    682,      1]], device='cuda:0')

In [ ]:
input_ids=input_tokenized[:,:-1].to(device)
input_ids

tensor([[     2,    105,   2364,    107,   4335,   1891,    506,   4553,    618,
          11446,   1936,    653,  88007,    532,   8082,   3217, 236761,    108,
          11919, 236787,  89099, 236888,   1599,    735,   2810,    496,  31386,
         236770, 236771, 236764, 236771, 236771, 236771,   9593,   7963,   3733,
         236761,  11738,   1590,    531,   3539,   1492, 236888,    106,    107,
            105,   4368,    107, 236782,    623,   2491,   1083,    623,   6120,
           1936,    827,    623,  27388,   1083,    623,   2805,   6169, 152718,
          13833,    607,  60055,   1781,    682]], device='cuda:0')

In [ ]:
target_ids=input_tokenized[:,1:].to(device)
target_ids

tensor([[   105,   2364,    107,   4335,   1891,    506,   4553,    618,  11446,
           1936,    653,  88007,    532,   8082,   3217, 236761,    108,  11919,
         236787,  89099, 236888,   1599,    735,   2810,    496,  31386, 236770,
         236771, 236764, 236771, 236771, 236771,   9593,   7963,   3733, 236761,
          11738,   1590,    531,   3539,   1492, 236888,    106,    107,    105,
           4368,    107, 236782,    623,   2491,   1083,    623,   6120,   1936,
            827,    623,  27388,   1083,    623,   2805,   6169, 152718,  13833,
            607,  60055,   1781,    682,      1]], device='cuda:0')

In [ ]:
import torch.nn as nn
def calculate_loss(logits,labels):
  loss_fn=nn.CrossEntropyLoss(reduction="none")
  cross_entropy = loss_fn(logits.view(-1, logits.shape[-1]), labels.view(-1))
  return cross_entropy

In [ ]:
from torch.optim import AdamW
model.train()

optimizer = AdamW(model.parameters(),lr=3e-5, weight_decay=0.01)

for _ in range(15):
  out=model(input_ids)
  loss = calculate_loss(out.logits,target_ids).mean()
  loss.backward()
  optimizer.step()
  optimizer.zero_grad()
  print(loss.item())

6.21875
3.234375
2.0
1.2890625
0.890625
0.64453125
0.470703125
0.353515625
0.291015625
0.265625
0.2490234375
0.2392578125
0.2314453125
0.224609375
0.21875


# Completed Fine tuning


In [ ]:
input_prompt=[
    {"role": "user", "content": "Classify the email as SPAM or HAM and explain why.\n\nEmail: Congratulations! You have won a ₹10,000 Amazon gift card. Click here to claim now!"},
]
encoded_input = tokenizer.apply_chat_template(
    conversation=input_prompt,
    return_tensors="pt",
    tokenize=True,
).to(device)

output = model.generate(input_ids=encoded_input["input_ids"], attention_mask=encoded_input["attention_mask"], max_new_tokens=25)
print(tokenizer.batch_decode(output ,skip_special_tokens=False))

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


['<bos><start_of_turn>user\nClassify the email as SPAM or HAM and explain why.\n\nEmail: Congratulations! You have won a ₹10,000 Amazon gift card. Click here to claim now!<end_of_turn>\n<start_of_turn>model\n{ "label": "SPAM", "reason": "Unsolicited reward with urgency." }<eos>']


# Save Model to Local

In [ ]:
model.save_pretrained("spam-gemma-model")
tokenizer.save_pretrained("spam-gemma-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('spam-gemma-model/tokenizer_config.json',
 'spam-gemma-model/chat_template.jinja',
 'spam-gemma-model/tokenizer.json')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("spam-gemma-model")
tokenizer = AutoTokenizer.from_pretrained("spam-gemma-model")

model.eval()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("spam_model", 'zip', "spam-gemma-model")
files.download("spam_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Load and Process Dataset from Hugging Face

In [ ]:
pip install datasets

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the dataset from Hugging Face
dataset = load_dataset("nvidia/HelpSteer")

# Display dataset structure
print(dataset)


README.md:   0%|          | 0.00/7.90k [00:00<?, ?B/s]

train.jsonl.gz: reconstructing file:   0%|          |  0.00B / 15.9MB            

train.jsonl.gz: downloading bytes:           |  0.00B            

validation.jsonl.gz: reconstructing file:   0%|          |  0.00B /  813kB            

validation.jsonl.gz: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/35331 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1789 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response', 'helpfulness', 'correctness', 'coherence', 'complexity', 'verbosity'],
        num_rows: 35331
    })
    validation: Dataset({
        features: ['prompt', 'response', 'helpfulness', 'correctness', 'coherence', 'complexity', 'verbosity'],
        num_rows: 1789
    })
})


In [ ]:
# Access the 'train' split if available and convert to pandas DataFrame
# Assuming the user wants to standardize the 'train' split. Adjust if another split is needed.
if 'train' in dataset:
    df = dataset['train'].to_pandas()
    print("\nFirst 5 rows of the dataset:")
    display(df.head())
    print("\nDataset Info:")
    df.info()

    # Example of a simple standardization: selecting a relevant column (e.g., 'instruction')
    # If 'standardize' implies a more complex transformation, please specify.
    if 'instruction' in df.columns:
        standardized_df = df[['instruction', 'response', 'context_id']].copy()
        print("\nFirst 5 rows of the 'standardized' DataFrame (selected columns):")
        display(standardized_df.head())

        # Save the standardized DataFrame to a CSV file
        output_filename = "helpsteer_standardized.csv"
        standardized_df.to_csv(output_filename, index=False)
        print(f"\nStandardized data saved to {output_filename}")
    else:
        print("The 'instruction' column was not found. Saving the full DataFrame as 'helpsteer_full.csv'")
        df.to_csv("helpsteer_full.csv", index=False)
        print(f"Full dataset saved to helpsteer_full.csv")
else:
    print("No 'train' split found in the dataset. Please specify which split to standardize.")



First 5 rows of the dataset:


,prompt,response,helpfulness,correctness,coherence,complexity,verbosity
0,What are the three most important things to co...,To build an assistive device to help an elderl...,3,4,4,2,2
1,What are the three most important things to co...,There are many different types of assistive de...,4,3,3,2,3
2,What are the three most important things to co...,When deciding what technology to use to build ...,4,4,4,2,2
3,What are the three most important things to co...,You can create an assistant device to help an ...,3,3,3,2,3
4,Background:\n<start of reference>\nFamily doct...,"Hi there! I'm Dr. Family, and I'm here to tell...",3,3,3,2,1



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35331 entries, 0 to 35330
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   prompt       35331 non-null  object
 1   response     35331 non-null  object
 2   helpfulness  35331 non-null  int32 
 3   correctness  35331 non-null  int32 
 4   coherence    35331 non-null  int32 
 5   complexity   35331 non-null  int32 
 6   verbosity    35331 non-null  int32 
dtypes: int32(5), object(2)
memory usage: 1.2+ MB
The 'instruction' column was not found. Saving the full DataFrame as 'helpsteer_full.csv'
Full dataset saved to helpsteer_full.csv


## Prepare Data for OpenAI LoRA Fine-tuning

In [ ]:
import json

# Ensure the 'train' split is available and converted to a pandas DataFrame
# This assumes `df` from the previous cell is the DataFrame to work with.
if 'train' in dataset:
    # If df was already created and is not empty
    if 'df' not in locals() or df.empty:
        df = dataset['train'].to_pandas()

    output_jsonl_filename = "helpsteer_openai_lora_finetune.jsonl"

    # Process each conversation and write to a JSONL file
    with open(output_jsonl_filename, 'w') as f:
        for index, row in df.iterrows():
            messages = []

            # Use 'prompt' as user and 'response' as assistant as these columns are present
            if 'prompt' in row and 'response' in row:
                messages.append({"role": "user", "content": row['prompt']})
                messages.append({"role": "assistant", "content": row['response']})
            else:
                print(f"Warning: Skipping row {index} due to missing 'prompt' or 'response' data.")
                continue

            if messages:
                json_line = json.dumps({"messages": messages})
                f.write(json_line + '\n')

    print(f"Data standardized for OpenAI LoRA fine-tuning and saved to {output_jsonl_filename}")

else:
    print("No 'train' split found in the dataset to prepare for OpenAI fine-tuning.")

Streaming output truncated to the last 5000 lines.
Data standardized for OpenAI LoRA fine-tuning and saved to helpsteer_openai_lora_finetune.jsonl
